In [9]:
import pandas as pd
import numpy as np
from nptdms import TdmsFile
from pathlib import Path
import os

import matplotlib.pyplot as plt
import scipy
from scipy import signal

import plotly.offline as py
import plotly.graph_objs as go
import plotly.io as pio

py.init_notebook_mode(connected=True)

from tqdm import tqdm
import warnings

%matplotlib inline

In [10]:
SAMPLING_FREQ = 12800

In [3]:
cut_intervals = dict()
params_excel = pd.read_excel('../input/raw_data/alu_v2_verification/alu_v2_parameters_acc.xlsx', 
                            skiprows=1)
def parse(row):
    cutno = row['Kesim']
    start, stop = map(int, row['cut-interval'].split(' ')[0].split('-'))
    cut_intervals[cutno]={'start':start, 'stop':stop}
params_excel.apply(parse, axis=1)

cut_intervals

{1: {'start': 3, 'stop': 16},
 2: {'start': 3, 'stop': 15},
 3: {'start': 2, 'stop': 10},
 4: {'start': 3, 'stop': 11},
 5: {'start': 1, 'stop': 7},
 6: {'start': 3, 'stop': 8}}

In [11]:
class ChatterData():
    def __init__(self, kanalno, cut=True):
        self.kanalno = kanalno
        self.outputpath = OUTPUTPATH.format(self.kanalno)
        
#         tdms_acc:pd.DataFrame = TdmsFile(DATAPATH/f"kanal{kanalno}_ivme.tdms").as_dataframe().rename(columns={"/'IVME'/'Untitled'":"acc"})
        
        tdms_acc:pd.DataFrame = TdmsFile(DATAPATH/f"kanal{kanalno}_ivme.tdms").as_dataframe().rename(columns={"/'IVME'/'Untitled'":"acc"})
        tdms_sound:pd.DataFrame = TdmsFile(DATAPATH/f"kanal{kanalno}_ses.tdms").as_dataframe().rename(columns={"/'SES'/'Untitled'":"acoustic"})
#         tdms_sound:pd.DataFrame = TdmsFile(DATAPATH/f"kanal{kanalno}_ivme.tdms").as_dataframe().rename(columns={"/'IVME'/'Untitled'":"acoustic"})
            
        self.tdms_data = pd.DataFrame(data={'time':np.arange(tdms_acc.shape[0])*(1/SAMPLING_FREQ),
                                'acc':tdms_acc['acc'],
                              'acoustic':tdms_sound['acoustic'],
                              })
        
        if cut:
            self.cut_interval = cut_intervals[self.kanalno]
            self.tdms_data = self.tdms_data.loc[(self.tdms_data['time']>=self.cut_interval['start'])&
                                                (self.tdms_data['time']<=self.cut_interval['stop'])]
            self.tdms_data.reset_index(drop=True, inplace=True)
        
        
        self.data_chunks = {}
        for i in range(int(self.tdms_data['time'].min()), int(self.tdms_data['time'].max())):
            self.data_chunks[i] = self.tdms_data.loc[(i<=self.tdms_data['time'])&
                                                     (self.tdms_data['time']<i+1)]
               
        self.fft_ylim = {'acc':0.0042, 'acoustic':0.05}
        
    def plot_timeseries(self, data, i, whichone='acc', fig=None, save=True, plot=True):
                
        trace = go.Scatter(x=data['time'], y=data[whichone], name=whichone)

        data = [trace]
        layout = go.Layout(
            xaxis={'title': 'Time(sec)'},
            yaxis={'title': 'Amplitude'}
        )

        fig = go.Figure(data=data, layout=layout)

        if save:
            inner_output_path = os.path.join(self.outputpath,f"{i}")
            os.makedirs(inner_output_path, exist_ok=True)
            pio.write_image(fig, os.path.join(inner_output_path,f"plot_{whichone}.png"))
            if HTML_OUTPUT:
                pio.write_html(fig, os.path.join(inner_output_path,f"plot_{whichone}.html"))
        if plot:
            py.iplot(fig)
    
    
    
    @staticmethod
    def fft(data, whichone):
        Fs = SAMPLING_FREQ # sampling freq
        Ts = 1/Fs # sampling interval
        t = data['time'] # time vector

        n = data.shape[0] # data lenght
        k = np.arange(n)
        T = n/Fs
        freq = k/T

        S = data[whichone].values
        S_fft = np.fft.fft(S)


        freq = freq[:n//2] # one side frequency range
        freq = freq[freq<=2500] # cut unnecessary frequencies
        
        S_fft = S_fft.real[:freq.shape[0]] / n # fft computing and normalization
        S_fft = np.abs(S_fft) # y-axis symetric correction
        
        fft_data = S_fft
        fft_freqs = freq
        return fft_data, fft_freqs
        
        
    def plot_fft(self,  data, i, whichone='acc', fig=None, save=True, plot=True):
        
        
        fft_data, fft_freqs = self.fft(data=data, whichone=whichone)

        if (fft_data.max() > self.fft_ylim[whichone]) and WARN:
            warnings.warn(f"fft_ylim should be increased to {fft_data.max()} from {self.fft_ylim[whichone]}")

        trace = go.Scatter(x=fft_freqs, y=fft_data, name='fft')

        data = [trace]
        layout = go.Layout(
            xaxis={'title': 'Freq(Hz)'},
            yaxis={'title': 'Amplitude', 'range':[0, self.fft_ylim[whichone]]}
        )

        fig = go.Figure(data=data, layout=layout)

        if save:
            inner_output_path = os.path.join(self.outputpath,f"{i}")
            os.makedirs(inner_output_path, exist_ok=True)
            pio.write_image(fig, os.path.join(inner_output_path,f"plot_fft_{whichone}.png"))
            if HTML_OUTPUT:
                pio.write_html(fig, os.path.join(inner_output_path,f"plot_fft_{whichone}.html"))
        if plot:
            py.iplot(fig)
    
    @staticmethod
    def spectrogram(data, whichone='acc'):
       
        S = data[whichone]
        spect_freqs, spect_times, spectrogram_data = signal.spectrogram(S, SAMPLING_FREQ)
        spect_freqs = spect_freqs[spect_freqs<=2500]  # cut unnecessary frequencies
        spect_times = spect_times
        spectrogram_data = spectrogram_data[:spect_freqs.shape[0], :]
        
        return spect_freqs, spect_times, spectrogram_data

    def plot_spectrogram(self, data, i, whichone='acc', fig=None, save=True, plot=True, csv=True):
        
  
        spect_freqs, spect_times, spectrogram_data = self.spectrogram(data=data, whichone=whichone)
        trace = go.Heatmap(z=10*np.log10(spectrogram_data), x=spect_times, y=spect_freqs, name='spectrogram')

        data = [trace]
        layout = go.Layout(
            xaxis={'title': 'Time(sec)'},
            yaxis={'title': 'Freq(Hz)'})

        fig = go.Figure(data=data, layout=layout)

        if save:
            inner_output_path = os.path.join(self.outputpath,f"{i}")
            os.makedirs(inner_output_path, exist_ok=True)
            pio.write_image(fig, os.path.join(inner_output_path,f"plot_spectrogram_{whichone}.png"))
            if HTML_OUTPUT:
                pio.write_html(fig, os.path.join(inner_output_path,f"plot_spectrogram_{whichone}.html"))
            if csv:
                np.savetxt(os.path.join(inner_output_path,f"spectrogram_{whichone}.csv"),
                           fig['data'][0]['z'], delimiter=',')

        if plot:
            py.iplot(fig)
        
        
    def save(self):
        
        os.makedirs(self.outputpath, exist_ok=True)
#         save full timeseries and full plots
        for whichone in ['acc', 'acoustic']:
            self.plot_timeseries(self.tdms_data, '00', whichone=whichone, plot=False, save=True)
            self.plot_fft(self.tdms_data, '00', whichone=whichone, plot=False, save=True)
            self.plot_spectrogram(self.tdms_data, '00', whichone=whichone, plot=False, save=True, csv=True)
        os.makedirs(os.path.join(self.outputpath,"00"), exist_ok=True)
        self.tdms_data.to_csv(os.path.join(self.outputpath,"00", f"data.csv"))
        
        # save each 1 sec timeseries and plots
        for key, data in self.data_chunks.items():
            for whichone in ['acc', 'acoustic']:
                self.plot_timeseries(data, str(key+1).zfill(2), whichone=whichone, plot=False, save=True)
                self.plot_fft(data, str(key+1).zfill(2), whichone=whichone, plot=False, save=True)
                self.plot_spectrogram(data, str(key+1).zfill(2), whichone=whichone, plot=False, save=True, csv=True)
            os.makedirs(os.path.join(self.outputpath,str(key+1).zfill(2)), exist_ok=True)
            data.to_csv(os.path.join(self.outputpath,str(key+1).zfill(2), f"data.csv"))


In [16]:
DATAPATH = Path("../input/raw_data/alu_v1")
OUTPUTPATH = "../input/preprocessed_data/alu_v1/kanal{}"
# REQUIRED_KANALNOS = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25]
# REQUIRED_KANALNOS = list(range(1, 65))
REQUIRED_KANALNOS = list(range(1, 27))
HTML_OUTPUT = False
WARN = False
for kanalno in tqdm(REQUIRED_KANALNOS):
    chatter_data = ChatterData(kanalno=kanalno, cut=False)
    chatter_data.save()

100%|██████████████████████████████████████████████████████████████████████████████████| 26/26 [07:36<00:00, 16.80s/it]


In [13]:
DATAPATH = Path("../input/raw_data/alu_v1")
OUTPUTPATH = "../input/preprocessed_data/alu_v1/kanal{}"
HTML_OUTPUT = False
c = ChatterData(kanalno=1, cut=False)

In [14]:
c.save()